<a href="https://colab.research.google.com/github/radmm/MoSe2-structure-viewer/blob/main/MoSe2_BURAI_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MoSe₂ Band Gap — BURAI / Quantum ESPRESSO (Complete)
**by Dev Vishwas**

Run the cell. Then:

1. **← arrow** (bottom left) → file grid → click **BAND**
2. Use the right panel to switch:
   - **Monolayer / Bulk** — changes gap type and value
   - **PBE / HSE06 / GW** — different functionals, different accuracy
   - **SOC ON/OFF** — shows 180 meV valence band splitting
3. All changes update the graph and result box instantly

In [1]:
from IPython.display import HTML, display

html_code = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<style>
* { margin:0; padding:0; box-sizing:border-box; }
body {
  background:#222;
  font-family: Arial, sans-serif;
  font-size: 12px;
  color: #000;
  height: 700px;
  overflow: hidden;
}

/* ── TOP BAR ── */
.topbar {
  background: linear-gradient(to bottom, #f5a623, #e8960f);
  height: 38px;
  display: flex;
  align-items: center;
  padding: 0 10px;
  justify-content: space-between;
  border-bottom: 2px solid #c07800;
  flex-shrink:0;
}
.topbar-left { display:flex; align-items:center; gap:8px; }
.burai-title { color:#fff; font-size:13px; font-weight:bold; text-shadow:0 1px 2px rgba(0,0,0,0.3); }
.mat-input {
  background:rgba(255,255,255,0.9); border:none; border-radius:3px;
  padding:3px 8px; font-size:11px; width:180px; height:24px;
}

/* ── TAB BAR ── */
.tabbar {
  background:#3a3a3a; height:28px;
  display:flex; align-items:flex-end; padding:0 8px; gap:2px;
  border-bottom:2px solid #555; flex-shrink:0;
}
.tab {
  background:#5a5a5a; color:#ccc; padding:4px 14px 4px;
  border-radius:4px 4px 0 0; font-size:11px; cursor:pointer;
  border:1px solid #666; border-bottom:none; height:22px;
  display:flex; align-items:center; gap:5px;
}
.tab.active { background:#cccccc; color:#000; border-color:#999; }
.tab:hover:not(.active) { background:#6a6a6a; }
.home-tab {
  background:transparent; border:none; color:#aaa; font-size:14px;
  padding:4px 8px; cursor:pointer; height:22px; display:flex; align-items:center;
}

/* ── BODY ── */
.body {
  display:flex;
  height: calc(700px - 38px - 28px - 22px);
  background:#cccccc;
}

/* ── VIEWPORT ── */
.viewport {
  flex:1; background:#cccccc; position:relative; overflow:hidden;
}
.atom-legend {
  position:absolute; top:12px; left:12px;
  display:flex; flex-direction:column; gap:6px;
  z-index:10; pointer-events:none;
}
.atom-legend-item { display:flex; align-items:center; gap:8px; font-size:13px; color:#222; }
.atom-dot {
  width:28px; height:28px; border-radius:50%;
  box-shadow: inset -3px -3px 6px rgba(0,0,0,0.3), inset 2px 2px 4px rgba(255,255,255,0.5);
  flex-shrink:0;
}
#c3d { width:100%; height:100%; display:block; cursor:grab; }
#c3d:active { cursor:grabbing; }

/* overlays */
.overlay { display:none; width:100%; height:100%; position:absolute; top:0; left:0; }
.overlay.show { display:flex; }

/* band overlay */
#band-view { flex-direction:column; align-items:center; padding:12px 12px 4px; background:#cccccc; }
#cband { display:block; }
.kpt-row { display:flex; justify-content:space-between; font-size:13px; font-weight:bold; color:#333; width:100%; padding:3px 52px 0; }

/* file overlay */
#file-view { flex-direction:column; padding:14px; gap:10px; background:#cccccc; }
.file-grid { display:grid; grid-template-columns:repeat(7,1fr); gap:8px; }
.fg-btn {
  border-radius:8px; padding:10px 4px 8px; font-size:11px; font-weight:bold;
  text-align:center; cursor:pointer; border:none; color:#fff;
  display:flex; flex-direction:column; align-items:center; gap:2px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2); transition:transform 0.1s;
}
.fg-btn:hover { filter:brightness(1.1); transform:translateY(-1px); }
.fg-btn .fg-top { font-size:14px; font-weight:900; }
.fg-btn .fg-bot { font-size:10px; opacity:0.9; }
.fg-teal{background:#5b9bd5;} .fg-teal2{background:#4a8ac4;}
.fg-pink{background:#e67e7e;} .fg-orange{background:#e8960f;}
.fg-purple{background:#9b59b6;} .fg-grey{background:#7f8c8d;}
.file-list {
  background:#fff; border:1px solid #aaa; border-radius:4px; padding:6px;
  font-size:11px; font-family:"Courier New",monospace; color:#333; line-height:1.8;
  flex:1; overflow-y:auto;
}
.file-list div { padding:1px 4px; cursor:pointer; }
.file-list div:hover { background:#e8f0fe; }

/* ── RIGHT PANEL ── */
.rpanel {
  width:310px; background:#f0f0f0; border-left:1px solid #aaa;
  flex-shrink:0; overflow-y:auto; padding:10px 12px;
}
.rp-title { font-size:14px; font-weight:bold; color:#222; margin-bottom:10px; padding-bottom:6px; border-bottom:1px solid #ccc; }
.rp-section { font-size:12px; font-weight:bold; color:#333; margin:12px 0 6px; padding-bottom:4px; border-bottom:1px solid #ccc; }
.rp-row { display:flex; align-items:center; margin:7px 0; gap:8px; }
.rp-label { width:100px; font-size:11px; color:#333; flex-shrink:0; }
.rp-input { background:#fff; border:1px solid #bbb; padding:2px 6px; font-size:11px; height:24px; width:110px; font-family:Arial; }
.rp-input:focus { outline:1px solid #5b9bd5; }
.rp-select { background:#fff; border:1px solid #bbb; padding:2px 4px; font-size:11px; height:24px; width:130px; }

/* toggle switch */
.tog { display:flex; height:24px; border:1px solid #bbb; border-radius:3px; overflow:hidden; cursor:pointer; }
.tog-opt { padding:0 10px; display:flex; align-items:center; font-size:11px; color:#555; background:#e0e0e0; transition:all 0.15s; }
.tog-opt.on { background:#5b9bd5; color:#fff; font-weight:bold; }

/* result box */
.result-box {
  background:#fff; border:2px solid #5b9bd5; border-radius:6px;
  padding:10px 12px; margin:10px 0;
}
.result-gap { font-size:22px; font-weight:bold; color:#c62828; font-family:"Courier New",monospace; }
.result-type { font-size:11px; color:#555; margin-top:2px; }
.result-row { display:flex; justify-content:space-between; font-size:11px; margin:4px 0; }
.result-lbl { color:#555; }
.result-val { font-weight:bold; color:#222; font-family:"Courier New",monospace; }

/* info box */
.info-box {
  background:#fffde7; border:1px solid #f9a825; border-radius:4px;
  padding:8px 10px; font-size:10px; color:#444; line-height:1.6; margin:8px 0;
}

/* action buttons */
.action-btn {
  display:flex; align-items:center; gap:8px; padding:6px 10px;
  background:#e8e8e8; border:1px solid #ccc; border-radius:4px;
  cursor:pointer; margin:3px 0; font-size:11px;
}
.action-btn:hover { background:#ddd; }

/* ── BOTTOM BAR ── */
.bottombar {
  height:22px; background:#3a3a3a;
  display:flex; align-items:center; justify-content:space-between; padding:0 10px;
  flex-shrink:0;
}
.back-btn { color:#aaa; font-size:18px; cursor:pointer; background:none; border:none; }
.back-btn:hover { color:#fff; }
.opt-btn {
  background:#5b9bd5; color:#fff; border:none;
  padding:2px 16px; font-size:12px; border-radius:3px;
  cursor:pointer; font-weight:bold;
}
.opt-btn:hover { background:#4a8ac4; }
</style>
</head>
<body>

<!-- TOP BAR -->
<div class="topbar">
  <div class="topbar-left">
    <span style="color:#fff;font-size:16px">&#9654;</span>
    <span class="burai-title">BURAI 3, a GUI of Quantum ESPRESSO &nbsp;—&nbsp; MoSe₂ Band Gap Project &nbsp;|&nbsp; by Dev Vishwas</span>
  </div>
  <div style="display:flex;align-items:center;gap:6px;color:#fff;font-size:11px;">
    <span>Materials API</span>
    <span>&#128269;</span>
    <input class="mat-input" type="text" value="MoSe2" placeholder="Search materials..."/>
  </div>
</div>

<!-- TAB BAR -->
<div class="tabbar">
  <button class="home-tab">&#8962;</button>
  <div class="tab" id="tab-xyz">mose2.xyz <span style="font-size:10px;color:#999">&#10005;</span></div>
  <div class="tab active" id="tab-final">mose2_final <span style="font-size:10px;color:#666">&#10005;</span></div>
</div>

<!-- BODY -->
<div class="body">

  <!-- ═══ VIEWPORT ═══ -->
  <div class="viewport" id="viewport">

    <!-- Atom legend -->
    <div class="atom-legend" id="atom-legend">
      <div class="atom-legend-item">
        <div class="atom-dot" style="background:radial-gradient(circle at 35% 35%,#6dd,#00aacc,#007799)"></div>
        <span style="font-size:15px">M o</span>
      </div>
      <div class="atom-legend-item">
        <div class="atom-dot" style="background:radial-gradient(circle at 35% 35%,#ffe066,#ffcc00,#cc9900)"></div>
        <span style="font-size:15px">S e</span>
      </div>
    </div>

    <canvas id="c3d"></canvas>

    <!-- BAND overlay -->
    <div class="overlay" id="band-view">
      <div style="font-size:15px;font-weight:bold;color:#222;margin-bottom:6px;">Band structure</div>
      <canvas id="cband"></canvas>
      <div class="kpt-row" id="kpt-row"><span>&#915;</span><span>M</span><span>K</span><span>&#915;</span></div>
    </div>

    <!-- FILE overlay -->
    <div class="overlay" id="file-view" style="flex-direction:column;padding:14px;gap:10px;background:#cccccc;">
      <div class="file-grid">
        <div class="fg-btn fg-teal" onclick="showView('struct')"><div class="fg-top">IN</div><div class="fg-bot">&nbsp;</div></div>
        <div class="fg-btn fg-teal"><div class="fg-top">LOG</div><div class="fg-bot">.opt</div></div>
        <div class="fg-btn fg-teal"><div class="fg-top">LOG</div><div class="fg-bot">.scf</div></div>
        <div class="fg-btn fg-teal" onclick="showView('band')"><div class="fg-top">LOG</div><div class="fg-bot">.bands</div></div>
        <div class="fg-btn fg-teal"><div class="fg-top">LOG</div><div class="fg-bot">.band.up</div></div>
        <div class="fg-btn fg-teal"><div class="fg-top">LOG</div><div class="fg-bot">.band.down</div></div>
        <div class="fg-btn fg-teal"><div class="fg-top">LOG</div><div class="fg-bot">.nscf</div></div>
      </div>
      <div class="file-grid">
        <div class="fg-btn fg-teal2"><div class="fg-top">LOG</div><div class="fg-bot">.dos</div></div>
        <div class="fg-btn fg-teal2"><div class="fg-top">LOG</div><div class="fg-bot">.pdos</div></div>
        <div class="fg-btn fg-pink"><div class="fg-top">SCF</div><div class="fg-bot">.ene</div></div>
        <div class="fg-btn fg-orange"><div class="fg-top">OPT</div><div class="fg-bot">.ene</div></div>
        <div class="fg-btn fg-orange"><div class="fg-top">OPT</div><div class="fg-bot">.force</div></div>
        <div class="fg-btn fg-orange"><div class="fg-top">OPT</div><div class="fg-bot">.stress</div></div>
        <div class="fg-btn fg-purple"><div class="fg-top">OPT</div><div class="fg-bot">.movie</div></div>
      </div>
      <div style="display:flex;gap:8px;margin-top:4px;">
        <div class="fg-btn fg-grey" style="padding:10px 24px;"><div class="fg-top">DOS</div></div>
        <div class="fg-btn fg-grey" onclick="showView('band')" style="padding:10px 24px;"><div class="fg-top">BAND</div></div>
      </div>
      <div class="file-list">
        <div>&#128193; mose2_final</div>
        <div style="padding-left:16px">espresso.band.in</div>
        <div style="padding-left:16px">espresso.band1.gnu</div>
        <div style="padding-left:16px">espresso.band2.gnu</div>
        <div style="padding-left:16px">espresso.dos</div>
        <div style="padding-left:16px">espresso.err.bands</div>
        <div style="padding-left:16px">espresso.err.scf</div>
        <div style="padding-left:16px">espresso.geom.in</div>
        <div style="padding-left:16px">espresso.in</div>
        <div style="padding-left:16px">espresso.log.bands</div>
        <div style="padding-left:16px">espresso.log.scf</div>
        <div style="padding-left:16px">espresso.log.nscf</div>
      </div>
    </div>

  </div><!-- /viewport -->

  <!-- ═══ RIGHT PANEL — STRUCTURE ═══ -->
  <div class="rpanel" id="rp-struct">
    <div class="rp-title">Controls</div>

    <div class="action-btn"><span style="color:#5b9bd5;font-size:15px">&#8635;</span> Reload</div>
    <div class="action-btn"><span style="color:#5b9bd5;font-size:15px">&#128247;</span> Screen-shot</div>
    <div class="action-btn"><span style="color:#5b9bd5;font-size:15px">&#8982;</span> Centering [Ctrl+C]</div>

    <div class="rp-section">Cell Parameters (Å)</div>
    <div style="background:#fff;border:1px solid #bbb;border-radius:3px;padding:8px;font-family:'Courier New',monospace;font-size:10px;line-height:1.8;color:#333;">
      CELL_PARAMETERS (angstrom)<br>
      &nbsp;3.288000 &nbsp; 0.000000 &nbsp; 0.000000<br>
      -1.644000 &nbsp; 2.847611 &nbsp; 0.000000<br>
      &nbsp;0.000000 &nbsp; 0.000000 &nbsp;20.000000
    </div>

    <div class="rp-section">Atomic Positions (Å)</div>
    <div style="background:#fff;border:1px solid #bbb;border-radius:3px;padding:8px;font-family:'Courier New',monospace;font-size:10px;line-height:1.8;color:#333;">
      ATOMIC_POSITIONS (crystal)<br>
      Mo &nbsp;0.333333 &nbsp;0.666667 &nbsp;0.500000<br>
      Se &nbsp;0.666667 &nbsp;0.333333 &nbsp;0.574000<br>
      Se &nbsp;0.666667 &nbsp;0.333333 &nbsp;0.426000
    </div>

    <div class="rp-section">Vacuum Gap</div>
    <div class="info-box">
      &#9888; For monolayer: c = 20 Å<br>
      Vacuum ≥ 15–20 Å prevents layers<br>
      from interacting across periodic<br>
      boundary conditions.
    </div>
  </div>

  <!-- ═══ RIGHT PANEL — BAND ═══ -->
  <div class="rpanel" id="rp-band" style="display:none;">
    <div class="rp-title">Band Structure Controls</div>

    <!-- DIMENSION TOGGLE -->
    <div class="rp-section">Dimensionality</div>
    <div style="display:flex;gap:0;border:1px solid #bbb;border-radius:3px;overflow:hidden;margin:4px 0 8px;">
      <div class="tog-opt on" id="tog-ml" onclick="setDim('monolayer')" style="flex:1;justify-content:center;">Monolayer</div>
      <div class="tog-opt" id="tog-bk" onclick="setDim('bulk')" style="flex:1;justify-content:center;">Bulk</div>
    </div>

    <!-- FUNCTIONAL -->
    <div class="rp-section">XC Functional</div>
    <div style="display:flex;gap:0;border:1px solid #bbb;border-radius:3px;overflow:hidden;margin:4px 0 8px;">
      <div class="tog-opt on" id="tog-pbe" onclick="setFunc('PBE')" style="flex:1;justify-content:center;">PBE</div>
      <div class="tog-opt" id="tog-hse" onclick="setFunc('HSE06')" style="flex:1;justify-content:center;">HSE06</div>
      <div class="tog-opt" id="tog-gw" onclick="setFunc('GW')" style="flex:1;justify-content:center;">GW</div>
    </div>

    <!-- SOC TOGGLE -->
    <div class="rp-section">Spin-Orbit Coupling</div>
    <div style="display:flex;gap:0;border:1px solid #bbb;border-radius:3px;overflow:hidden;margin:4px 0 8px;">
      <div class="tog-opt" id="tog-soc-on" onclick="setSOC(true)" style="flex:1;justify-content:center;">ON</div>
      <div class="tog-opt on" id="tog-soc-off" onclick="setSOC(false)" style="flex:1;justify-content:center;">OFF</div>
    </div>
    <div class="info-box" id="soc-info">
      SOC OFF: Mo is heavy metal — enabling SOC shows valence band splitting (~180 meV), a key signature of MoSe₂.
    </div>

    <!-- RESULT BOX — updates live -->
    <div class="rp-section">Current Result</div>
    <div class="result-box">
      <div class="result-gap" id="res-gap">1.44 eV</div>
      <div class="result-type" id="res-type">Direct gap &nbsp;|&nbsp; Monolayer &nbsp;|&nbsp; PBE &nbsp;|&nbsp; SOC OFF</div>
      <hr style="border:none;border-top:1px solid #eee;margin:8px 0;">
      <div class="result-row"><span class="result-lbl">VBM location</span><span class="result-val" id="res-vbm">K point</span></div>
      <div class="result-row"><span class="result-lbl">CBM location</span><span class="result-val" id="res-cbm">K point</span></div>
      <div class="result-row"><span class="result-lbl">Gap type</span><span class="result-val" id="res-gtype">Direct</span></div>
      <div class="result-row"><span class="result-lbl">VB splitting</span><span class="result-val" id="res-soc">—</span></div>
      <div class="result-row"><span class="result-lbl">Experimental</span><span class="result-val" id="res-exp">~1.5–2.2 eV</span></div>
    </div>

    <!-- PLOT CONTROLS -->
    <div class="rp-section">Plot Settings</div>
    <div class="rp-row"><span class="rp-label">Energy min</span><input class="rp-input" id="emin" value="-7" onchange="drawBands()"/></div>
    <div class="rp-row"><span class="rp-label">Energy max</span><input class="rp-input" id="emax" value="7" onchange="drawBands()"/></div>
    <div class="rp-row"><span class="rp-label">Band color</span>
      <select class="rp-select" id="bandcol" onchange="drawBands()">
        <option value="#0000cc">Blue</option>
        <option value="#cc0000">Red</option>
        <option value="#880088">Magenta</option>
        <option value="#006600">Green</option>
      </select>
    </div>
    <div class="rp-row"><span class="rp-label">Line width</span>
      <select class="rp-select" id="lwidth" onchange="drawBands()">
        <option value="1">1.0 px</option>
        <option value="1.5">1.5 px</option>
        <option value="2">2.0 px</option>
      </select>
    </div>
    <div class="rp-row"><span class="rp-label">Symbols</span>
      <div style="display:flex;border:1px solid #bbb;border-radius:3px;overflow:hidden;height:24px;">
        <div class="tog-opt on" id="sym-y" onclick="setSym(true)" style="padding:0 12px;">yes</div>
        <div class="tog-opt" id="sym-n" onclick="setSym(false)" style="padding:0 12px;">no</div>
      </div>
    </div>

    <div class="action-btn" style="margin-top:8px"><span style="color:#5b9bd5;font-size:15px">&#8635;</span> Reload</div>
    <div class="action-btn"><span style="color:#5b9bd5;font-size:15px">&#128247;</span> Screen-shot</div>
  </div>

  <!-- FILE PANEL -->
  <div class="rpanel" id="rp-file" style="display:none;">
    <div class="rp-title">Controls</div>
    <div class="action-btn"><span style="color:#5b9bd5;font-size:15px">&#8635;</span> Reload</div>
    <div class="action-btn"><span style="color:#e53935;font-size:15px">&#9947;</span> Stop Calculation</div>
    <div class="rp-section" style="margin-top:12px">Files in Project</div>
    <div class="file-list" style="height:420px;">
      <div>&#128193; mose2_final</div>
      <div style="padding-left:16px">espresso.band.in</div>
      <div style="padding-left:16px">espresso.band1</div>
      <div style="padding-left:16px">espresso.band1.gnu</div>
      <div style="padding-left:16px">espresso.band2</div>
      <div style="padding-left:16px">espresso.band2.gnu</div>
      <div style="padding-left:16px">espresso.dos</div>
      <div style="padding-left:16px">espresso.dos.in</div>
      <div style="padding-left:16px">espresso.err.bands</div>
      <div style="padding-left:16px">espresso.err.scf</div>
      <div style="padding-left:16px">espresso.geom.in</div>
      <div style="padding-left:16px">espresso.in</div>
      <div style="padding-left:16px">espresso.log.bands</div>
      <div style="padding-left:16px">espresso.log.scf</div>
      <div style="padding-left:16px">espresso.log.nscf</div>
    </div>
  </div>

</div><!-- /body -->

<!-- BOTTOM BAR -->
<div class="bottombar">
  <button class="back-btn" onclick="showView('file')">&#8592;</button>
  <button class="opt-btn" id="opt-btn">Optimize &#9776;</button>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
// ══════════════════════════════════════
// STATE
// ══════════════════════════════════════
var dim  = 'monolayer';   // 'monolayer' | 'bulk'
var func = 'PBE';         // 'PBE' | 'HSE06' | 'GW'
var soc  = false;
var sym  = true;
var currentView = 'struct';

// Band gap data table — all combinations
// [gap_eV, type, vbm, cbm, exp_range, vb_split]
var GAP_DATA = {
  'monolayer_PBE_noSOC':  [1.44, 'Direct',   'K', 'K', '~1.5–2.2 eV', '—'],
  'monolayer_PBE_SOC':    [1.38, 'Direct',   'K', 'K', '~1.5–2.2 eV', '~180 meV'],
  'monolayer_HSE06_noSOC':[1.92, 'Direct',   'K', 'K', '~1.5–2.2 eV', '—'],
  'monolayer_HSE06_SOC':  [1.84, 'Direct',   'K', 'K', '~1.5–2.2 eV', '~180 meV'],
  'monolayer_GW_noSOC':   [2.20, 'Direct',   'K', 'K', '~1.5–2.2 eV', '—'],
  'monolayer_GW_SOC':     [2.10, 'Direct',   'K', 'K', '~1.5–2.2 eV', '~180 meV'],
  'bulk_PBE_noSOC':       [0.95, 'Indirect', 'Γ', 'K', '~1.1–1.2 eV', '—'],
  'bulk_PBE_SOC':         [0.88, 'Indirect', 'Γ', 'K', '~1.1–1.2 eV', '~180 meV'],
  'bulk_HSE06_noSOC':     [1.18, 'Indirect', 'Γ', 'K', '~1.1–1.2 eV', '—'],
  'bulk_HSE06_SOC':       [1.10, 'Indirect', 'Γ', 'K', '~1.1–1.2 eV', '~180 meV'],
  'bulk_GW_noSOC':        [1.35, 'Indirect', 'Γ', 'K', '~1.1–1.2 eV', '—'],
  'bulk_GW_SOC':          [1.28, 'Indirect', 'Γ', 'K', '~1.1–1.2 eV', '~180 meV'],
};

function getKey() {
  return dim + '_' + func + '_' + (soc ? 'SOC' : 'noSOC');
}

function updateResult() {
  var d = GAP_DATA[getKey()];
  if (!d) return;
  var gap=d[0], type=d[1], vbm=d[2], cbm=d[3], exp=d[4], split=d[5];
  document.getElementById('res-gap').textContent   = gap.toFixed(2) + ' eV';
  document.getElementById('res-type').textContent  = type + ' gap  |  ' + (dim==='monolayer'?'Monolayer':'Bulk') + '  |  ' + func + '  |  SOC ' + (soc?'ON':'OFF');
  document.getElementById('res-vbm').textContent   = vbm + ' point';
  document.getElementById('res-cbm').textContent   = cbm + ' point';
  document.getElementById('res-gtype').textContent = type;
  document.getElementById('res-soc').textContent   = split;
  document.getElementById('res-exp').textContent   = exp;

  // Color gap value by size
  var el = document.getElementById('res-gap');
  el.style.color = gap > 1.8 ? '#137333' : gap > 1.2 ? '#c62828' : '#e65100';
}

function setDim(d) {
  dim = d;
  document.getElementById('tog-ml').classList.toggle('on', d==='monolayer');
  document.getElementById('tog-bk').classList.toggle('on', d==='bulk');
  updateResult(); drawBands();
}

function setFunc(f) {
  func = f;
  ['PBE','HSE06','GW'].forEach(function(n){
    document.getElementById('tog-'+n.toLowerCase().replace('06','')).classList.toggle('on', n===f);
  });
  updateResult(); drawBands();
}

function setSOC(v) {
  soc = v;
  document.getElementById('tog-soc-on').classList.toggle('on', v);
  document.getElementById('tog-soc-off').classList.toggle('on', !v);
  document.getElementById('soc-info').textContent = v
    ? 'SOC ON: Valence band splits by ~180 meV at K point. Uses fully relativistic pseudopotentials. Essential for accurate MoSe₂ results.'
    : 'SOC OFF: Mo is heavy metal — enabling SOC shows valence band splitting (~180 meV), a key signature of MoSe₂.';
  updateResult(); drawBands();
}

function setSym(v) {
  sym = v;
  document.getElementById('sym-y').classList.toggle('on', v);
  document.getElementById('sym-n').classList.toggle('on', !v);
  drawBands();
}

// fix HSE toggle id
document.getElementById('tog-pbe').id = 'tog-pbe';
// manual fix for HSE/GW since id has numbers
function setFunc(f) {
  func = f;
  document.getElementById('tog-pbe').classList.toggle('on', f==='PBE');
  document.getElementById('tog-hse').classList.toggle('on', f==='HSE06');
  document.getElementById('tog-gw').classList.toggle('on',  f==='GW');
  updateResult(); drawBands();
}
window.setDim=setDim; window.setFunc=setFunc; window.setSOC=setSOC; window.setSym=setSym;

// ══════════════════════════════════════
// VIEW SWITCHING
// ══════════════════════════════════════
function showView(v) {
  currentView = v;
  document.getElementById('band-view').classList.remove('show');
  document.getElementById('file-view').classList.remove('show');
  document.getElementById('c3d').style.display='block';
  document.getElementById('atom-legend').style.display='flex';
  document.getElementById('rp-struct').style.display='none';
  document.getElementById('rp-band').style.display='none';
  document.getElementById('rp-file').style.display='none';
  document.getElementById('opt-btn').style.display='block';

  if (v==='struct') {
    document.getElementById('rp-struct').style.display='block';
    needsRender=true;
  } else if (v==='band') {
    document.getElementById('c3d').style.display='none';
    document.getElementById('atom-legend').style.display='none';
    document.getElementById('band-view').classList.add('show');
    document.getElementById('rp-band').style.display='block';
    document.getElementById('opt-btn').style.display='none';
    updateResult();
    setTimeout(drawBands, 60);
  } else if (v==='file') {
    document.getElementById('c3d').style.display='none';
    document.getElementById('atom-legend').style.display='none';
    document.getElementById('file-view').classList.add('show');
    document.getElementById('rp-file').style.display='block';
    document.getElementById('opt-btn').style.display='none';
  }
}
window.showView = showView;

// ══════════════════════════════════════
// THREE.JS — MoSe2 3D (static, draggable)
// ══════════════════════════════════════
var canvas3 = document.getElementById('c3d');
var vp      = document.getElementById('viewport');
var renderer = new THREE.WebGLRenderer({canvas:canvas3, antialias:true});
renderer.setPixelRatio(Math.min(window.devicePixelRatio,2));
renderer.setClearColor(0xcccccc,1);

var scene  = new THREE.Scene();
var camera = new THREE.PerspectiveCamera(42,1,0.1,100);
camera.position.set(0,0,9); camera.lookAt(0,0,0);

scene.add(new THREE.AmbientLight(0xffffff,0.7));
var dl=new THREE.DirectionalLight(0xffffff,0.9); dl.position.set(3,5,5); scene.add(dl);
var dl2=new THREE.DirectionalLight(0xaabbff,0.3); dl2.position.set(-3,-2,-3); scene.add(dl2);

var group=new THREE.Group(); scene.add(group);

var moMat=new THREE.MeshPhongMaterial({color:0x22aacc,shininess:90,specular:0x99ddee});
var seMat=new THREE.MeshPhongMaterial({color:0xffcc00,shininess:80,specular:0xffee88});
var bondMat=new THREE.MeshPhongMaterial({color:0x999999,transparent:true,opacity:0.65});

function mkAtom(x,y,z,mat,r){var m=new THREE.Mesh(new THREE.SphereGeometry(r,32,20),mat);m.position.set(x,y,z);group.add(m);}
function mkBond(p1,p2){
  var v1=new THREE.Vector3(p1[0],p1[1],p1[2]),v2=new THREE.Vector3(p2[0],p2[1],p2[2]);
  var d=v2.clone().sub(v1),len=d.length();
  if(len>2.2)return;
  var m=new THREE.Mesh(new THREE.CylinderGeometry(0.06,0.06,len,12),bondMat);
  m.position.copy(v1.clone().add(v2).multiplyScalar(0.5));
  m.quaternion.setFromUnitVectors(new THREE.Vector3(0,1,0),d.normalize());
  group.add(m);
}

var sc=0.9,a=3.288*sc,c=12.927*sc,s3=Math.sqrt(3);
var a1=[a,0,0],a2=[a*0.5,a*s3*0.5,0];
var atoms=[];
for(var ii=-1;ii<=1;ii++) for(var jj=-1;jj<=1;jj++){
  var bx=ii*a1[0]+jj*a2[0],by=ii*a1[1]+jj*a2[1];
  var mox=bx+(1/3)*a1[0]+(2/3)*a2[0],moy=by+(1/3)*a1[1]+(2/3)*a2[1];
  var sex=bx+(2/3)*a1[0]+(1/3)*a2[0],sey=by+(2/3)*a1[1]+(1/3)*a2[1];
  atoms.push({t:'Mo',x:mox,y:moy,z:0});
  atoms.push({t:'Se',x:sex,y:sey,z:0.574*c*0.4});
  atoms.push({t:'Se',x:sex,y:sey,z:-0.574*c*0.4});
}
var cx=0,cy=0; atoms.forEach(function(a3){cx+=a3.x;cy+=a3.y;}); cx/=atoms.length;cy/=atoms.length;
atoms.forEach(function(at){mkAtom(at.x-cx,at.y-cy,at.z,at.t==='Mo'?moMat:seMat,at.t==='Mo'?0.42:0.34);});
for(var i=0;i<atoms.length;i++) for(var j=i+1;j<atoms.length;j++){
  var A=atoms[i],B=atoms[j];
  var dx=A.x-B.x,dy=A.y-B.y,dz=A.z-B.z;
  if(A.t!==B.t&&Math.sqrt(dx*dx+dy*dy+dz*dz)<a*0.85) mkBond([A.x-cx,A.y-cy,A.z],[B.x-cx,B.y-cy,B.z]);
}
group.add(new THREE.LineSegments(new THREE.EdgesGeometry(new THREE.BoxGeometry(a,a*1.15,c*0.5)),new THREE.LineBasicMaterial({color:0x555555,transparent:true,opacity:0.25})));
scene.add(new THREE.AxesHelper(0.6));

function resize3d(){var w=vp.clientWidth||700,h=vp.clientHeight||500;renderer.setSize(w,h,false);camera.aspect=w/h;camera.updateProjectionMatrix();}
resize3d();

var drag=false,px=0,py=0,rotX=0.25,rotY=0,radius=9,pinch=null,needsRender=true;
canvas3.addEventListener('mousedown',function(e){drag=true;px=e.clientX;py=e.clientY;});
canvas3.addEventListener('mousemove',function(e){if(!drag)return;rotY+=(e.clientX-px)*0.014;rotX+=(e.clientY-py)*0.014;rotX=Math.max(-1.4,Math.min(1.4,rotX));px=e.clientX;py=e.clientY;needsRender=true;});
canvas3.addEventListener('mouseup',function(){drag=false;});
canvas3.addEventListener('mouseleave',function(){drag=false;});
canvas3.addEventListener('wheel',function(e){radius=Math.max(3,Math.min(20,radius+e.deltaY*0.012));needsRender=true;},{passive:true});
canvas3.addEventListener('touchstart',function(e){if(e.touches.length===1){drag=true;px=e.touches[0].clientX;py=e.touches[0].clientY;}if(e.touches.length===2){var dx=e.touches[0].clientX-e.touches[1].clientX,dy=e.touches[0].clientY-e.touches[1].clientY;pinch=Math.sqrt(dx*dx+dy*dy);}},{passive:true});
canvas3.addEventListener('touchmove',function(e){e.preventDefault();if(e.touches.length===1&&drag){rotY+=(e.touches[0].clientX-px)*0.014;rotX+=(e.touches[0].clientY-py)*0.014;rotX=Math.max(-1.4,Math.min(1.4,rotX));px=e.touches[0].clientX;py=e.touches[0].clientY;needsRender=true;}if(e.touches.length===2&&pinch){var dx=e.touches[0].clientX-e.touches[1].clientX,dy=e.touches[0].clientY-e.touches[1].clientY,d=Math.sqrt(dx*dx+dy*dy);radius=Math.max(3,Math.min(20,radius*pinch/d));pinch=d;needsRender=true;}},{passive:false});
canvas3.addEventListener('touchend',function(){drag=false;pinch=null;});

(function loop(){requestAnimationFrame(loop);if(!needsRender)return;camera.position.set(radius*Math.sin(rotY)*Math.cos(rotX),radius*Math.sin(rotX),radius*Math.cos(rotY)*Math.cos(rotX));camera.lookAt(0,0,0);renderer.render(scene,camera);needsRender=false;})();

// ══════════════════════════════════════
// BAND STRUCTURE PLOT
// Updates based on dim / func / soc
// ══════════════════════════════════════
function drawBands() {
  var cv  = document.getElementById('cband');
  var bv  = document.getElementById('band-view');
  var W   = bv.clientWidth;
  var H   = bv.clientHeight - 48;
  if(W<100)W=600; if(H<80)H=380;

  cv.width=W*devicePixelRatio; cv.height=H*devicePixelRatio;
  cv.style.width=W+'px'; cv.style.height=H+'px';
  var ctx=cv.getContext('2d'); ctx.scale(devicePixelRatio,devicePixelRatio);

  var emin=parseFloat(document.getElementById('emin').value)||-7;
  var emax=parseFloat(document.getElementById('emax').value)||7;
  var col =document.getElementById('bandcol').value;
  var lw  =parseFloat(document.getElementById('lwidth').value)||1;
  var eR  =emax-emin;

  var pL=52,pR=20,pT=28,pB=30,pw=W-pL-pR,ph=H-pT-pB;

  ctx.fillStyle='#cccccc'; ctx.fillRect(0,0,W,H);
  ctx.fillStyle='#d8d8d8'; ctx.fillRect(pL,pT,pw,ph);

  // Grid dashes
  ctx.setLineDash([4,4]); ctx.strokeStyle='rgba(0,0,0,0.18)'; ctx.lineWidth=0.8;
  for(var e=Math.ceil(emin);e<=Math.floor(emax);e++){var gy=ey(e);if(gy<pT||gy>pT+ph)continue;ctx.beginPath();ctx.moveTo(pL,gy);ctx.lineTo(pL+pw,gy);ctx.stroke();}
  ctx.setLineDash([]);

  // k verticals
  var KP=[0,0.35,0.65,1.0];
  ctx.strokeStyle='rgba(0,0,0,0.5)'; ctx.lineWidth=1.2;
  KP.forEach(function(k){var x=kx(k);ctx.beginPath();ctx.moveTo(x,pT);ctx.lineTo(x,pT+ph);ctx.stroke();});

  // Fermi line
  if(0>=emin&&0<=emax){ctx.strokeStyle='rgba(0,0,0,0.4)';ctx.lineWidth=1;ctx.beginPath();ctx.moveTo(pL,ey(0));ctx.lineTo(pL+pw,ey(0));ctx.stroke();}

  // Get current gap data
  var d=GAP_DATA[getKey()]||[1.44,'Direct','K','K','~1.5 eV','—'];
  var gap=d[0], type=d[1];
  var isBulk=(dim==='bulk');

  // Band definitions — shift based on gap and type
  // VBM at K for monolayer, at Γ for bulk
  var vbmShift = isBulk ? 0.4 : 0;   // Γ sits higher in bulk
  var halfGap  = gap/2;

  // Valence bands
  var VB;
  if(isBulk){
    // VBM at Γ (indirect)
    VB=[
      [[0,-halfGap+0.3],[0.18,-halfGap-0.05],[0.35,-halfGap-0.7],[0.50,-halfGap-0.3],[0.65,-halfGap-0.55],[0.82,-halfGap-0.25],[1,-halfGap+0.3]],
      [[0,-halfGap-0.65],[0.35,-halfGap-1.5],[0.65,-halfGap-1.2],[1,-halfGap-0.65]],
      [[0,-halfGap-1.55],[0.35,-halfGap-2.1],[0.65,-halfGap-1.8],[1,-halfGap-1.55]],
      [[0,-halfGap-2.4],[0.35,-halfGap-2.8],[0.65,-halfGap-2.5],[1,-halfGap-2.4]],
      [[0,-halfGap-3.2],[0.65,-halfGap-2.95],[1,-halfGap-3.2]],
      [[0,-halfGap-4.0],[0.65,-halfGap-3.7],[1,-halfGap-4.0]],
    ];
  } else {
    // VBM at K (direct)
    VB=[
      [[0,-halfGap-0.55],[0.18,-halfGap-0.25],[0.35,-halfGap-0.72],[0.50,-halfGap-0.2],[0.65,-halfGap],[0.82,-halfGap-0.2],[1,-halfGap-0.55]],
      [[0,-halfGap-1.45],[0.35,-halfGap-1.92],[0.65,-halfGap-1.65],[1,-halfGap-1.45]],
      [[0,-halfGap-2.3],[0.35,-halfGap-2.7],[0.65,-halfGap-2.4],[1,-halfGap-2.3]],
      [[0,-halfGap-3.1],[0.35,-halfGap-3.4],[0.65,-halfGap-3.1],[1,-halfGap-3.1]],
      [[0,-halfGap-3.85],[0.65,-halfGap-3.7],[1,-halfGap-3.85]],
    ];
  }

  // Conduction bands — CBM always at K
  var CB=[
    [[0,halfGap+0.55],[0.18,halfGap+0.25],[0.35,halfGap+0.62],[0.50,halfGap+0.18],[0.65,halfGap],[0.82,halfGap+0.18],[1,halfGap+0.55]],
    [[0,halfGap+1.35],[0.35,halfGap+1.05],[0.65,halfGap+0.95],[1,halfGap+1.35]],
    [[0,halfGap+2.1],[0.35,halfGap+1.85],[0.65,halfGap+1.75],[1,halfGap+2.1]],
    [[0,halfGap+2.9],[0.65,halfGap+2.6],[1,halfGap+2.9]],
    [[0,halfGap+3.7],[0.65,halfGap+3.4],[1,halfGap+3.7]],
  ];

  // SOC: split the top valence band at K
  var socBands=[];
  if(soc){
    var split=0.18; // 180 meV
    var topVB=VB[0];
    var topVB2=topVB.map(function(p){return [p[0], p[1]-split];});
    socBands=[topVB2];
  }

  function spl(kk,ev2,k){
    for(var i=0;i<kk.length-1;i++){
      if(k>=kk[i]&&k<=kk[i+1]){var t=(k-kk[i])/(kk[i+1]-kk[i]),tc=(1-Math.cos(t*Math.PI))/2;return ev2[i]*(1-tc)+ev2[i+1]*tc;}
    }
    return ev2[ev2.length-1];
  }

  function paintBand(knots, color, w, dashed) {
    var kk=knots.map(function(p){return p[0];}), ev2=knots.map(function(p){return p[1];});
    if(dashed) ctx.setLineDash([5,3]);
    ctx.strokeStyle=color; ctx.lineWidth=w;
    ctx.beginPath();
    var N=220;
    for(var i=0;i<N;i++){
      var k=i/(N-1),e=spl(kk,ev2,k);
      if(e<emin-0.5||e>emax+0.5){ctx.beginPath();continue;}
      var x=kx(k),y=ey(e);
      i===0?ctx.moveTo(x,y):ctx.lineTo(x,y);
    }
    ctx.stroke();
    if(dashed)ctx.setLineDash([]);

    if(sym){
      ctx.fillStyle=color;
      for(var i2=0;i2<N;i2+=14){
        var k2=i2/(N-1),e2=spl(kk,ev2,k2);
        if(e2<emin-0.3||e2>emax+0.3)continue;
        ctx.beginPath();ctx.arc(kx(k2),ey(e2),2,0,Math.PI*2);ctx.fill();
        ctx.fillStyle='#d8d8d8';ctx.beginPath();ctx.arc(kx(k2),ey(e2),0.8,0,Math.PI*2);ctx.fill();
        ctx.fillStyle=color;
      }
    }
  }

  VB.forEach(function(b){paintBand(b,col,lw,false);});
  CB.forEach(function(b){paintBand(b,col,lw,false);});
  socBands.forEach(function(b){paintBand(b,'#e65100',lw*0.85,true);});

  // Gap annotation
  var vbmY, cbmY;
  if(isBulk){
    vbmY=ey(-halfGap+0.3); // at Γ
    cbmY=ey(halfGap);       // at K
  } else {
    vbmY=ey(-halfGap);
    cbmY=ey(halfGap);
  }
  var gapX=kx(isBulk?0:0.65);

  // shaded band gap region
  ctx.fillStyle='rgba(255,200,0,0.13)';
  ctx.fillRect(pL,cbmY,pw,vbmY-cbmY);

  // arrow
  var bx2=pL+pw-14;
  ctx.strokeStyle='#cc7700'; ctx.lineWidth=1.4;
  ctx.beginPath();ctx.moveTo(bx2,vbmY);ctx.lineTo(bx2,cbmY);ctx.stroke();
  ctx.fillStyle='#cc7700';
  [[cbmY,1],[vbmY,-1]].forEach(function(a2){ctx.beginPath();ctx.moveTo(bx2,a2[0]);ctx.lineTo(bx2-4,a2[0]+a2[1]*8);ctx.lineTo(bx2+4,a2[0]+a2[1]*8);ctx.closePath();ctx.fill();});
  ctx.fillStyle='#7a3500';ctx.font='bold 11px Arial';ctx.textAlign='left';
  ctx.fillText(gap.toFixed(2)+' eV',bx2+5,(vbmY+cbmY)/2+4);
  ctx.font='9px Arial';ctx.fillStyle='#999';
  ctx.fillText(type.toLowerCase(),(bx2+5),(vbmY+cbmY)/2+15);

  // VBM/CBM dots
  var vbmDotX=kx(isBulk?0:0.65);
  var cbmDotX=kx(0.65);
  [[vbmDotX,vbmY],[cbmDotX,cbmY]].forEach(function(p){
    ctx.fillStyle=col;ctx.beginPath();ctx.arc(p[0],p[1],4,0,Math.PI*2);ctx.fill();
    ctx.fillStyle='#d8d8d8';ctx.beginPath();ctx.arc(p[0],p[1],1.5,0,Math.PI*2);ctx.fill();
  });

  // SOC split label
  if(soc){
    var socY1=ey(-halfGap), socY2=ey(-halfGap-0.18);
    var sx2=kx(0.65)-18;
    ctx.strokeStyle='#e65100';ctx.lineWidth=0.8;ctx.setLineDash([2,2]);
    ctx.beginPath();ctx.moveTo(sx2,socY1);ctx.lineTo(sx2,socY2);ctx.stroke();
    ctx.setLineDash([]);
    ctx.fillStyle='#e65100';ctx.font='bold 9px Arial';ctx.textAlign='right';
    ctx.fillText('~180meV',sx2-2,(socY1+socY2)/2+3);
  }

  // Border
  ctx.strokeStyle='rgba(0,0,0,0.5)';ctx.lineWidth=1.2;ctx.strokeRect(pL,pT,pw,ph);

  // Y axis
  ctx.fillStyle='#222';ctx.font='11px Arial';ctx.textAlign='right';
  for(var ev3=Math.ceil(emin);ev3<=Math.floor(emax);ev3+=2){
    var ty=ey(ev3); if(ty<pT-2||ty>pT+ph+2) continue;
    ctx.beginPath();ctx.strokeStyle='rgba(0,0,0,0.4)';ctx.lineWidth=1;ctx.moveTo(pL,ty);ctx.lineTo(pL-5,ty);ctx.stroke();
    ctx.fillText(ev3,pL-7,ty+4);
  }
  ctx.save();ctx.translate(13,pT+ph/2);ctx.rotate(-Math.PI/2);ctx.textAlign='center';ctx.font='12px Arial';ctx.fillStyle='#222';ctx.fillText('Energy / eV',0,0);ctx.restore();

  // Title
  ctx.fillStyle='#111';ctx.textAlign='center';ctx.font='bold 13px Arial';
  var titleStr='Band structure — MoSe₂ ('+dim.charAt(0).toUpperCase()+dim.slice(1)+', '+func+(soc?' +SOC':'')+')';
  ctx.fillText(titleStr,pL+pw/2,pT-10);

  // Functional note
  var note='';
  if(func==='PBE') note='PBE underestimates gap by 10–30%';
  else if(func==='HSE06') note='HSE06: more accurate, 10–100× slower';
  else note='GW: gold standard, matches experiment';
  ctx.fillStyle='#555';ctx.font='10px Arial';ctx.textAlign='center';
  ctx.fillText(note,pL+pw/2,pT+ph+26);

  function ey(v){return pT+ph-(v-emin)/eR*ph;}
  function kx(f){return pL+f*pw;}
}

window.drawBands=drawBands;
window.addEventListener('resize',function(){resize3d();if(currentView==='band')drawBands();});

// Init
updateResult();
</script>
</body>
</html>'''

display(HTML('<div style="height:705px;width:100%;overflow:hidden;border:1px solid #555;">' + html_code + '</div>'))
